In [43]:

import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

import datasets, sys
print(f"versão dataset: {datasets.__version__}")

# Um limite amostral para os prints não ficarem enormes
lim_print = 5

versão dataset: 2.18.0


#### Função `average_pool`

<span style="font-size: 0.85em">

Reduz a saída token-a-token do modelo a um único vetor por sentença, ignorando os tokens de padding.

#### Parâmetros

**`last_hidden_states`** — tensor de saída da última camada do modelo, com shape `(batch, seq_len, hidden)`.

**`attention_mask`** — tensor que define quais tokens são reais (valor `1`) e quais devem ser ignorados (valor `0`). Shape `(batch, seq_len)`: uma linha por sequência do batch, uma posição por token.

#### Conceitos

**Tensor** — array multidimensional de números.

**Shape** — tupla com um valor por dimensão, indicando o tamanho de cada uma:

| Dimensões | Exemplo | Shape |
|---|---|---|
| 0 (escalar) | `3.14` | `()` |
| 1 (vetor) | `[1, 2, 3]` | `(3,)` |
| 2 (matriz) | `[[1, 2], [3, 4]]` | `(2, 2)` |
| 3 (cubo) | `[[[1,2],[3,4]], [[5,6],[7,8]]]` | `(2, 2, 2)` |
| N | sem nome próprio, mesma ideia | — |

#### Como funciona

Cada token vira um vetor de tamanho `hidden` (768, 1024 — depende do modelo), e a média é tirada **posição a posição**: a dimensão 0 do vetor final é a média das dimensões 0 de todos os tokens, e assim por diante.

O resultado tem exatamente o mesmo tamanho de um vetor de token individual, mas agora representa a sentença inteira.

</span>

In [3]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

#### Carregamento do modelo

<span style="font-size: 0.85em">

```python
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
```

**`AutoTokenizer`** — converte texto em IDs numéricos. Devolve `input_ids` (os tokens) e `attention_mask` (quais posições são reais).

**`AutoModel`** — a rede em si, sem cabeça de tarefa. Devolve `last_hidden_state` com shape `(batch, seq_len, 768)`.

**`Auto*`** — classes genéricas: leem a configuração do repositório e instanciam a arquitetura correta (aqui, XLM-RoBERTa) sem que você precise nomeá-la.

**`from_pretrained(...)`** — baixa os pesos do Hugging Face Hub na primeira execução e os guarda em cache local; nas seguintes, carrega do disco.

**`multilingual-e5-base`** — modelo de embeddings multilíngue (inclui português), `hidden = 768`. Usa mean pooling, daí a `average_pool`.
</span>

In [4]:
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7945.42it/s]


#### Carregamento e tokenização dos documentos

<span style="font-size: 0.85em">

#### `load_dataset`

**`'unicamp-dl/mmarco'`** — versão multilíngue do MS MARCO, corpus padrão de recuperação de documentos.

**`'collection-portuguese'`** — a *config*: seleciona a coleção de passagens em português. É o acervo de textos, não muda entre treino e teste.

**`streaming=True`** — não baixa o dataset inteiro (são milhões de passagens); os exemplos chegam sob demanda. O retorno é um `IterableDataset`, que não aceita indexação por posição.

**`trust_remote_code=True`** — autoriza a execução do script `mmarco.py` do repositório. Exige `datasets < 4.0`; nas versões novas, loading scripts foram removidos.

#### Amostragem

**`.take(100)`** — pega os 100 primeiros exemplos do stream. O `list(...)` materializa o iterador em memória.

**`item['text']`** — extrai só o texto de cada registro, descartando o `id`.

#### `tokenizer`

**`max_length=512`** — limite de tokens por sequência.

**`truncation=True`** — corta o que passar desse limite.

**`padding=True`** — completa as sequências curtas até o comprimento da maior do batch, para formar um tensor retangular. É o que cria os paddings que a `average_pool` precisa ignorar.

**`return_tensors='pt'`** — devolve tensores PyTorch em vez de listas.

O resultado é um dicionário com `input_ids` e `attention_mask`, ambos com shape `(100, seq_len)`.

</span>

In [45]:
# Passa os parâmetros do dataset a ser buscado
dataset_docs = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', streaming=True, trust_remote_code=True)

# Pega só os primeiros 100 exemplos
docs = list(dataset_docs['collection'].take(100))
print(f"Exemplo docs:\n{docs[0]}\n")

# Separa o texto do id e labels
collection_textos = [f"{item['text']}" for item in docs]
print(f"Exemplo texto:\n{collection_textos[0]}\n")

# transforma de texto para token e depois em id e attention mask
docs_batch = tokenizer(collection_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')
print(f"ids:\n{docs_batch['input_ids'][:lim_print, :lim_print]}\n")
print(f"máscara de atenção:\n{docs_batch['attention_mask'][:lim_print, :lim_print]}")

Repo card metadata block was not found. Setting CardData to empty.


Exemplo docs:
{'id': 0, 'text': 'A presença de comunicação entre mentes científicas foi tão importante para o sucesso do Projeto Manhattan quanto o intelecto científico. A única nuvem que paira sobre a impressionante conquista dos pesquisadores e engenheiros atômicos é o que seu sucesso realmente significou; centenas de milhares de vidas inocentes destruídas.'}

Exemplo texto:
A presença de comunicação entre mentes científicas foi tão importante para o sucesso do Projeto Manhattan quanto o intelecto científico. A única nuvem que paira sobre a impressionante conquista dos pesquisadores e engenheiros atômicos é o que seu sucesso realmente significou; centenas de milhares de vidas inocentes destruídas.

ids:
tensor([[     0,     62,  83541,      8,  98146],
        [     0,    180, 121926, 147037,     28],
        [     0,    357,    433,    846,   1028],
        [     0,    180, 121926, 147037,   1615],
        [     0,    493,  50364,      8,   2124]])

máscara de atenção:
tensor([[1, 1

#### Carregamento e tokenização das consultas

<span style="font-size: 0.85em">

**`'queries-portuguese'`** — config com as consultas do MS MARCO, as queries são as perguntas para as "passage".

**`['train']`** — acessa os dados definidos para treino. Datasets costumam vir separados em train, validation e test para que treino e avaliação usem dados distintos.

**Prefixo `"query: "`** — obrigatório no E5, e distinto do `"passage: "` usado nos documentos. É assim que o modelo diferencia os dois papéis; sem isso o ranking degrada.

**`max_length=512`** — herdado do batch de documentos, mas consultas são curtas (poucas palavras). Como `padding=True` preenche até a maior sequência *do batch*, e não até 512, não há desperdício.

O resultado tem a mesma estrutura do `docs_batch`: `input_ids` e `attention_mask` com shape `(100, seq_len)`.

</span>

In [46]:
# Passa os parâmetros do dataset a ser buscado
dataset_queries = load_dataset('unicamp-dl/mmarco', 'queries-portuguese', streaming=True, trust_remote_code=True)

# Pega só os primeiros 100 exemplos
queries = list(dataset_queries['train'].take(100))
print(f"Exemplo queries:\n{queries[1]}\n")

# Separa o texto.
queries_textos = [f"{item['text']}" for item in queries]
print(f"Exemplo query:\n{queries_textos[1]}\n")

# transforma de texto para token e depois em id e attention mask
queries_batch = tokenizer(queries_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')
print(f"ids:\n{queries_batch['input_ids'][:lim_print, :lim_print]}\n")
print(f"máscara de atenção:\n{queries_batch['attention_mask'][:lim_print, :lim_print]}")

Repo card metadata block was not found. Setting CardData to empty.


Exemplo queries:
{'id': 634306, 'text': 'o que significa bem móvel no histórico de crédito'}

Exemplo query:
o que significa bem móvel no histórico de crédito

ids:
tensor([[     0,  48106, 132038,      2,      1],
        [     0,     36,     41,  12330,   5289],
        [     0,   7002,   1715,     36,   3189],
        [     0,  47739,   7685,      8,  33780],
        [     0,     36,     41,    393,  14543]])

máscara de atenção:
tensor([[1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1]])


#### Geração dos embeddings contextuais das consultas

<span style="font-size: 0.85em">

**`model(**queries_batch)`** — o `**` desempacota o dicionário do tokenizer; equivale a `model(input_ids=..., attention_mask=...)`.

**`queries_outputs`** — objeto `BaseModelOutput`, acessível por atributo. O campo relevante é `last_hidden_state`, com shape `(100, seq_len, 768)`: um vetor por token.

**`average_pool(...)`** — reduz a `(100, 768)`: um vetor por consulta. A máscara é passada de novo porque o modelo não a devolve na saída — ele retorna vetores para todas as posições, inclusive os paddings.

**`F.normalize(..., p=2, dim=1)`** — normalização L2 ao longo da dimensão 1 (a das 768 features): cada vetor passa a ter comprimento 1. Com isso o produto interno vira cosseno, e a similaridade depende só da direção, não da magnitude.

**`(100,768)`** — 100 foi o número escolhido de textos(itens no batch) e 768 é o tamanho de cada vetor da arquitetura **`multilingual-e5-base`**. e o **`seq_len`**. é a quantidade de tokens de cada sequência do batch.

#### Fluxo dos shapes

```
last_hidden_state    (100, seq_len, 768)   um vetor por token
      ↓ average_pool
queries_embeddings   (100, 768)            um vetor por consulta
      ↓ F.normalize
queries_embeddings   (100, 768)            mesmo shape, comprimento 1
```

</span>

In [ ]:
# Calcula os embeddings contextuais de cada posição do batch
queries_outputs = model(**queries_batch)
print(f"Exemplo de um embedding contextual de uma query:\n{queries_outputs.last_hidden_state[0][:lim_print, :lim_print]}\n")

# Faz a média de todos os embeddings em um só tensor -> contexto geral da query
queries_embeddings = average_pool(queries_outputs.last_hidden_state, queries_batch['attention_mask'])
print(f"Exemplo média de todos os embeddings de uma query:\n{queries_embeddings[0][:lim_print]}\n")

# Normaliza os vetores (Normalização L2) para que todos tenham um "comprimento" (raiz da soma dos quadrados) igual a 1
# p=2 pois é a normalização L2 = euclidiana.
queries_embeddings = F.normalize(queries_embeddings, p=2, dim=1)
print(f"Exemplo de embedding normalizada:\n{queries_embeddings[0][:lim_print]}")

Exemplo de um embedding contextual de uma query:
tensor([[-0.0015,  0.3886, -0.0278, -0.0125,  0.0957],
        [-0.1981,  0.3106,  0.0151,  0.1930,  0.4064],
        [-0.1087,  0.3781, -0.0808,  0.2984,  0.0751],
        [-0.1120,  0.4233,  0.0679,  0.0373,  0.4476],
        [ 0.0205,  0.3841,  0.0346,  0.4197,  0.5480]],
       grad_fn=<SliceBackward0>)

Exemplo média de todos os embeddings de uma query:
tensor([-0.1051,  0.3752, -0.0064,  0.1290,  0.2562], grad_fn=<SliceBackward0>)

Exemplo de embedding normalizada:
tensor([-0.0064,  0.0229, -0.0004,  0.0079,  0.0156], grad_fn=<SliceBackward0>)


### Geração dos embeddings dos documentos

<span style="font-size: 0.85em">

Mesmo processo aplicado às consultas, agora sobre as passagens da coleção — o **mesmo modelo** e a **mesma normalização**, condição para que os dois conjuntos de vetores sejam comparáveis.

**`model(**docs_batch)`** — `last_hidden_state` com shape `(100, seq_len, 768)`. Aqui o `seq_len` é bem maior que o das consultas: passagens têm dezenas de tokens, consultas têm poucos. A diferença some no pooling.

**`average_pool(...)`** — reduz a `(100, 768)`: um vetor por passagem.

**`F.normalize(..., p=2, dim=1)`** — comprimento 1 em cada vetor, ele diz que a normalização é feita ao longo das 768 features, uma linha de cada vez. Se fosse dim=0, cada coluna teria comprimento 1 — o que misturaria as 100 consultas entre si e não faria sentido nenhum..

</span>

In [49]:
# Calcula os embeddings contextuais de cada posição do batch
docs_outputs = model(**docs_batch)
print(f"Exemplo de um embedding contextual de um texto:\n{docs_outputs.last_hidden_state[0][:lim_print, :lim_print]}\n")

# Faz a média de todos os embeddings em um só tensor -> contexto geral do texto
docs_embeddings = average_pool(docs_outputs.last_hidden_state, docs_batch['attention_mask'])
print(f"Exemplo média de todos os embeddings de um texto:\n{queries_embeddings[0][:lim_print]}\n")

# Normaliza os vetores (Normalização L2) para que todos tenham um "comprimento" (raiz da soma dos quadrados) igual a 1
docs_embeddings = F.normalize(docs_embeddings, p=2, dim=1)
print(f"Exemplo de embedding normalizada:\n{docs_embeddings[0][:lim_print]}")

Exemplo de um embedding contextual de um texto:
tensor([[-0.6357,  1.1537,  0.3374,  0.5907,  0.5854],
        [-0.6798,  1.4169, -0.0771,  0.6531,  0.9560],
        [-0.6899,  1.3160,  0.1435,  0.7221,  1.1289],
        [-0.4698,  1.1369, -0.0626,  0.8005,  0.8721],
        [-0.4703,  1.3623,  0.2218,  0.7894,  0.9574]],
       grad_fn=<SliceBackward0>)

Exemplo média de todos os embeddings de um texto:
tensor([-0.0064,  0.0229, -0.0004,  0.0079,  0.0156], grad_fn=<SliceBackward0>)

Exemplo de embedding normalizada:
tensor([-0.0288,  0.0693,  0.0060,  0.0407,  0.0435], grad_fn=<SliceBackward0>)
